# Child simulation — automated V&V

No out-of-the-box comparisons are available yet; the context is wired up so the
blockers are reproducible. See `../README.md` for the measure triage.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

from vivarium.validation import ValidationContext

from lsff_utils import paths

In [ ]:
LOCATION = "nigeria"
RUN = ""  # psimulate run timestamp; blank picks the newest complete run

location_root = paths.CHILD_RESULTS_ROOT / LOCATION
if RUN:
    RESULTS_DIR = location_root / RUN
else:
    # Require results/ too: psimulate writes model_specification.yaml at launch, so a
    # crashed or in-flight run would otherwise win on timestamp.
    RESULTS_DIR = max(
        (
            d
            for d in location_root.iterdir()
            if (d / "model_specification.yaml").exists() and (d / "results").is_dir()
        ),
        key=lambda d: d.name,
    )

REPORT_PATH = f"child_validation_report_{LOCATION}.html"
print(RESULTS_DIR)

In [ ]:
# psimulate names branch columns after the config-path leaf.
vc = ValidationContext(
    results_dir=RESULTS_DIR,
    scenario_columns=["child_scenario", "maternal_scenario"],
)

## Available simulation outputs

`person_time_total` will be absent — the sim's dataset is named `person_time`, which the
loader's `person_time_*` derivation does not match. Every ratio measure needs it.

In [ ]:
vc.get_sim_outputs()

## Available artifact keys

In [ ]:
vc.get_artifact_keys()

## Comparisons

In [ ]:
BASELINE = {"child_scenario": "baseline", "maternal_scenario": "baseline"}

# Blocked on total person time and on age-group alignment (README, Child blockers 1-2).
# vc.add_comparison(
#     "cause.all_causes.cause_specific_mortality_rate",
#     test_source="sim",
#     ref_source="artifact",
#     test_scenarios=BASELINE,
# )

sorted(vc.comparisons)

## Report

In [ ]:
vc.generate_results(output_path=REPORT_PATH)